# The Holographic Unfolding:
## Reconstructing Reality from a 1D Data Stream

**WILL Relational Geometry · Live Instrument**

---

In this computational proof, we synthesize a one-dimensional data stream — the time-evolution of light-frequency shifts, i.e. radial velocity — using the established first post-Newtonian (1PN) framework of General Relativity. This mimics the minimal data footprint obtainable from astronomical observations.

Our objective is radical: we test whether the entirety of a celestial system's physical reality — its absolute scale, its true mass, and its complete three-dimensional geometry — can be resurrected solely from this one-dimensional temporal stream.

Utilising the closed algebraic framework of WILL Relational Geometry (see [R.O.M. Equations, Part I](https://willrg.com/documents/WILL_RG_I.pdf#sec:rom)), we process this data and completely bypass both Newtonian constraints and the classical mass–inclination degeneracy. By extracting the absolute metric — the Schwarzschild radius — and the effective mass of the system from dimensionless phase shifts alone, we argue that physical reality is not a container of objects but a holographic projection of relational information.

---

[Complete derivation](https://willrg.com/documents/WILL_RG_I.pdf#sec:M_sin(i)) · [Colab notebook](https://colab.research.google.com/github/AntonRize/WILL/blob/main/Colab_Notebooks/ROM_HOLOGRAPHIC_REALITY_DECODER.ipynb) · [WILL-AI](https://willrg.com/will-ai/) · [Method limitations analisys](https://willrg.com/msini_test.html)

## Phase I: Synthetic Data Generation
In this stage, we simulate a high-eccentricity binary system with a massive central body. The simulation includes 1PN relativistic effects such as Schwarzschild precession, gravitational redshift, and transverse Doppler shifts.

In [7]:
!pip install emcee

#@title 🛰️ DATA SOURCE SELECTOR
#@markdown Choose whether to generate a new synthetic 1PN dataset or load your own custom CSV for testing.

DATA_MODE = "Synthetic" #@param ["Synthetic", "Custom"]
CUSTOM_FILE_NAME = "" #@param {type:"string"}
RANDOM_SEED = 14687 #@param {type:"integer"}

import numpy as np
import pandas as pd
import os


if DATA_MODE == "Synthetic":
    # ==========================================
    #  1PN DATA GENERATOR (STRICT BETA VALIDATION)
    # ==========================================
    np.random.seed(RANDOM_SEED)

    G = 6.67430e-11
    C = 299792458.0
    M_SUN = 1.98847e30
    SEC_PER_YEAR = 365.25 * 86400

    print(f"Initializing 1PN Generative Engine (Seed: {RANDOM_SEED})...")

    while True:
        log_M = np.random.uniform(np.log10(10.0), np.log10(1e9))
        M_bh = (10**log_M) * M_SUN
        P_scale = 10**(log_M / 3.0 - 2.0)
        P_yrs = np.random.uniform(1.0, 50.0) * P_scale
        if P_yrs < 0.01: P_yrs = 0.01
        e_true = np.random.uniform(0.85, 0.98)
        i_true = np.random.uniform(np.radians(10.0), np.radians(170.0))
        w_true = np.random.uniform(0.0, 2 * np.pi)
        vz0_true = np.random.uniform(-50.0, 50.0)
        T_peri = 58000.0
        NOISE_SIGMA_KMS = 3.0
        P_sec = P_yrs * SEC_PER_YEAR
        a_meters = (G * M_bh * (P_sec**2) / (4 * np.pi**2))**(1/3)
        beta_true = np.sqrt(G * M_bh / a_meters) / C

        t_bg = np.random.uniform(T_peri - (P_yrs * 365 * 0.5), T_peri + (P_yrs * 365 * 1.5), 40)
        t_peri_1 = np.random.normal(T_peri, P_yrs * 365 * 0.05, 60)
        t_peri_2 = np.random.normal(T_peri + P_yrs * 365.25, P_yrs * 365 * 0.05, 60)
        t_obs = np.sort(np.concatenate([t_bg, t_peri_1, t_peri_2]))

        M_unwrapped = (2 * np.pi / P_sec) * (t_obs * 86400 - T_peri * 86400)
        orbit_count = np.floor(M_unwrapped / (2 * np.pi))
        M_wrapped = M_unwrapped % (2 * np.pi)
        E = M_wrapped.copy()
        for _ in range(50):
            f = E - e_true * np.sin(E) - M_wrapped
            dE = f / (1 - e_true * np.cos(E))
            E -= dE
            if np.max(np.abs(dE)) < 1e-12: break

        nu_wrapped = 2 * np.arctan2(np.sqrt(1 + e_true) * np.sin(E / 2), np.sqrt(1 - e_true) * np.cos(E / 2))
        nu_positive = nu_wrapped % (2 * np.pi)
        nu_unwrapped = nu_positive + orbit_count * 2 * np.pi
        r_obs = a_meters * (1 - e_true**2) / (1 + e_true * np.cos(nu_positive))
        delta_w_per_radian = (3 * G * M_bh) / (a_meters * (1 - e_true**2) * C**2)
        w_dynamic = w_true + delta_w_per_radian * nu_unwrapped
        K_amp = np.sqrt(G * M_bh / (a_meters * (1 - e_true**2))) * np.sin(i_true)
        v_rad = K_amp * (np.cos(nu_positive + w_dynamic) + e_true * np.cos(w_dynamic))
        v_orb_sq_t = G * M_bh * (2.0 / r_obs - 1.0 / a_meters)
        z_grav = (G * M_bh) / (r_obs * C**2)
        z_transverse = v_orb_sq_t / (2 * C**2)
        v_rel_shift_kms = (z_grav + z_transverse) * C / 1000.0
        rel_shift_amplitude = np.max(v_rel_shift_kms) - np.min(v_rel_shift_kms)
        snr_rel = rel_shift_amplitude / NOISE_SIGMA_KMS

        if snr_rel >= 4.0 and beta_true >= 0.006: break

    rv_clean_kms = vz0_true + (v_rad / 1000.0) + v_rel_shift_kms
    rv_noisy_kms = rv_clean_kms + np.random.normal(0, NOISE_SIGMA_KMS, len(t_obs))

    pd.DataFrame({'MJD': t_obs, 'RV_km_s': rv_noisy_kms, 'sigma_km_s': np.full(len(t_obs), NOISE_SIGMA_KMS)}).to_csv('GR_BLIND_DATASET.csv', index=False)
    pd.DataFrame([{'P_yrs': P_yrs, 'e_true': e_true, 'w_true_deg': np.degrees(w_true) % 360, 'i_true_deg': np.degrees(i_true) % 180, 'vz0_true': vz0_true, 'M_true_solar': M_bh/M_SUN}]).to_csv('GR_BLIND_TRUTH.csv', index=False)
    print("STATUS: Synthetic data generated and saved as 'GR_BLIND_DATASET.csv'.")

else:
    if os.path.exists(CUSTOM_FILE_NAME):
        custom_df = pd.read_csv(CUSTOM_FILE_NAME)
        custom_df.to_csv('GR_BLIND_DATASET.csv', index=False)
        print(f"STATUS: Custom file '{CUSTOM_FILE_NAME}' loaded into workspace.")
    else:
        print(f"ERROR: File '{CUSTOM_FILE_NAME}' not found. Please upload it to the Colab file explorer.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.4/47.4 kB 2.3 MB/s eta 0:00:00
Initializing 1PN Generative Engine (Seed: 14687)...
STATUS: Synthetic data generated and saved as 'GR_BLIND_DATASET.csv'.


## Phase II: The Holographic Decoder (Inversion)
Using a multi-stage approach (Keplerian Scout → Global Sniper → MCMC), we invert the observed redshift signal to extract the underlying relational invariants: **Period (P)**, **Eccentricity (e)**, and the **Kinematic Projection (̢)**.

### Stage 3: MCMC Bayesian Mapping
In this final decoding stage, we employ a Markov Chain Monte Carlo (MCMC) sampler to map the multi-dimensional posterior probability distribution. This process does more than just find a 'best fit'; it rigorously tests the uniqueness of the relational solution and provides formal uncertainties for the absolute metric. By exploring the topology of the likelihood surface, we confirm that the 1D data stream contains a singular, non-degenerate path back to the 3D reality of the system.

In [8]:
# ==========================================
#  R.O.M. HOLGRAPHIC DECODER
# ==========================================

import numpy as np
import pandas as pd
from scipy.optimize import differential_evolution
import emcee
import warnings
warnings.filterwarnings('ignore')

# ---------------------------------------------------------
# CONSTANTS
# ---------------------------------------------------------
C_KMS = 299792.458 # Speed of light in km/s

# ---------------------------------------------------------
# 0. LOAD DATA
# ---------------------------------------------------------
print("Loading GR_BLIND_DATASET.csv...")
df = pd.read_csv('GR_BLIND_DATASET.csv')
t_obs = df['MJD'].values
vz_obs = df['RV_km_s'].values
sigma_vz = df['sigma_km_s'].values

Z_obs = 1.0 + (vz_obs / C_KMS)
sigma_Z = sigma_vz / C_KMS

idx_peak = np.argmax(np.abs(vz_obs))
t_peak = t_obs[idx_peak]

# ---------------------------------------------------------
# 1. R.O.M. EXACT GEOMETRY ENGINE (CONTINUOUS)
# ---------------------------------------------------------
def get_phase_unwrapped(t, t_peri, P, e):
    M_unwrapped = (2 * np.pi / P) * (t - t_peri)
    orbit_count = np.floor(M_unwrapped / (2 * np.pi))
    M_wrapped = M_unwrapped % (2 * np.pi)

    E = M_wrapped.copy()
    for _ in range(30):
        f = E - e * np.sin(E) - M_wrapped
        dE = f / (1 - e * np.cos(E))
        E -= dE
        if np.max(np.abs(dE)) < 1e-10:
            break

    o_wrapped = 2 * np.arctan2(np.sqrt(1 + e) * np.sin(E / 2), np.sqrt(1 - e) * np.cos(E / 2))
    o_positive = o_wrapped % (2 * np.pi)
    o_unwrapped = o_positive + orbit_count * 2 * np.pi
    return o_unwrapped

def generate_z_raw_dynamic(t_obs, o_unwrapped, beta, i_inc, beta_z0, e, omega_0, P_days, T_peri):
    # EXACT WILL RELATIONAL GEOMETRY ACCUMULATION (No truncation)
    # tau^2 = (1 - beta^2)(1 - kappa^2) where kappa^2 = 2*beta^2
    tau_sq = (1.0 - beta**2) * (1.0 - 2.0 * beta**2)

    # tau_Y^2 = 1 - tau^2 (Strict divergence including cross-coupling term)
    tau_Y_sq = 1.0 - tau_sq

    # Exact geometric phase shift per radian, normalized by shape factor
    omega_dynamic = omega_0 + (tau_Y_sq / (1.0 - e**2)) * o_unwrapped

    o = o_unwrapped % (2 * np.pi)

    # LOS Doppler Amplitude Calculation
    beta_int = beta / np.sqrt(1 - e**2)
    K = beta_int * np.sin(i_inc)
    beta_los = K * (np.cos(o + omega_dynamic) + e * np.cos(omega_dynamic))

    # Second-order systemic relational invariants (Independent of inclination)
    beta_o_sq = (beta**2) * (1 + e**2 + 2 * e * np.cos(o)) / (1 - e**2)
    kappa_o_sq = 2 * (beta**2) * (1 + e * np.cos(o)) / (1 - e**2)

    # --- PROTECT AGAINST UNPHYSICAL TOPOLOGY (NaN PREVENTION) ---
    if np.any(beta_o_sq >= 1.0) or np.any(kappa_o_sq >= 1.0):
        return np.full_like(t_obs, np.nan)

    Z_sys = (1 - beta_o_sq)**(-0.5) * (1 - kappa_o_sq)**(-0.5)

    return Z_sys * (1 + beta_los) * (1 + beta_z0)

# ---------------------------------------------------------
# 2. STAGE 1: KEPLERIAN SCOUT
# ---------------------------------------------------------
def classical_objective(params):
    K_guess, vz0_guess, e_guess, omega_guess, P_days_guess, t_peri_guess = params
    try:
        o_unwrapped = get_phase_unwrapped(t_obs, t_peri_guess, P_days_guess, e_guess)
        o_obs = o_unwrapped % (2 * np.pi)
        v_rad_kms = K_guess * (np.cos(o_obs + omega_guess) + e_guess * np.cos(omega_guess)) + vz0_guess
        Z_model_class = 1.0 + (v_rad_kms / C_KMS)
        return np.sum(((Z_obs - Z_model_class) / sigma_Z)**2)
    except:
        return 1e10

bounds_stage1 = [
    (10.0, 100000.0), (-150.0, 150.0), (0.50, 0.995),
    (0.0, 2 * np.pi), (10.0, 500000.0), (t_peak - 1000.0, t_peak + 1000.0)
]

print(f"STAGE 1: Keplerian Scout (Locating P, e, T_peri)...")
res_s1 = differential_evolution(classical_objective, bounds_stage1, strategy='best1bin', maxiter=1000, popsize=30, tol=1e-4, seed=101)
K_s1, vz0_s1, e_s1, omega_s1, P_s1, t_peri_s1 = res_s1.x
print(f"-> Skeleton Found: P = {P_s1/365.25:.2f} yrs, e = {e_s1:.4f}")

# --- DYNAMIC BETA ANCHOR (Prevents false peaks and speeds up processing) ---
beta_min_approx = (K_s1 / C_KMS) * np.sqrt(1 - e_s1**2)
beta_upper_bound = min(0.15, beta_min_approx * 15.0)

# ---------------------------------------------------------
# 3. STAGE 2: R.O.M. GLOBAL SNIPER
# ---------------------------------------------------------
def rom_objective(params):
    beta, i, vz0_c, e, omega, P, t_peri = params
    try:
        o_unwrapped = get_phase_unwrapped(t_obs, t_peri, P, e)
        Z_model = generate_z_raw_dynamic(t_obs, o_unwrapped, beta, i, vz0_c, e, omega, P, t_peri)
        if np.any(np.isnan(Z_model)): return 1e10
        return np.sum(((Z_obs - Z_model) / sigma_Z)**2)
    except:
        return 1e10

# Bounding inclination strictly to [0, pi/2] breaks the mirror degeneracy.
bounds_stage2 = [
    (0.00001, beta_upper_bound),                      # beta (Dynamically anchored)
    (0.001, np.pi/2),                                 # i (Folded domain)
    (-150.0/C_KMS, 150.0/C_KMS),                      # vz0_c
    (max(0.5, e_s1 - 0.05), min(0.999, e_s1 + 0.05)), # e (Constrained by S1)
    (0.0, 2 * np.pi),                                 # omega
    (max(10.0, P_s1 - 1000.0), P_s1 + 1000.0),        # P (Constrained by S1)
    (t_peri_s1 - 200.0, t_peri_s1 + 200.0)            # T_peri (Constrained by S1)
]

print("\nSTAGE 2: Full R.O.M. Global Sniper (Deterministic Decoupling)...")
res_s2 = differential_evolution(rom_objective, bounds_stage2, strategy='best1bin', maxiter=1000, popsize=20, tol=1e-5, seed=101)
beta_s2, i_s2, vz0_c_s2, e_s2, omega_s2, P_s2, t_peri_s2 = res_s2.x
print(f"-> Exact Peak Found: beta = {beta_s2:.6f}, i = {np.degrees(i_s2):.2f} deg, vz0 = {vz0_c_s2*C_KMS:.2f} km/s")

# ---------------------------------------------------------
# 4. STAGE 3: MCMC MAPPING
# ---------------------------------------------------------
def log_prior(theta):
    beta, i, vz0_c, e, omega, P, t_peri = theta
    if not (0.00001 < beta < beta_upper_bound): return -np.inf
    if not (0.0 < i < np.pi/2): return -np.inf
    if not (0.5 < e < 0.999): return -np.inf
    if not (0.0 < omega < 2*np.pi): return -np.inf
    if not (10.0 < P < 500000.0): return -np.inf
    if abs(vz0_c * C_KMS) > 150.0: return -np.inf
    return 0.0

def log_likelihood(theta):
    beta, i, vz0_c, e, omega, P, t_peri = theta
    try:
        o_unwrapped = get_phase_unwrapped(t_obs, t_peri, P, e)
        Z_model = generate_z_raw_dynamic(t_obs, o_unwrapped, beta, i, vz0_c, e, omega, P, t_peri)
        if np.any(np.isnan(Z_model)): return -np.inf
        return -0.5 * np.sum(((Z_obs - Z_model) / sigma_Z)**2)
    except:
        return -np.inf

def log_probability(theta):
    lp = log_prior(theta)
    if not np.isfinite(lp):
        return -np.inf
    return lp + log_likelihood(theta)

print("\nSTAGE 3: MCMC Bayesian Mapping (Running 1000 steps)...")
nwalkers = 32
ndim = 7

# Exact initialization centered on the verified Global Sniper peak.
# Added microscopic Gaussian jitter to ensure matrix full rank.
initial_pos = np.array([beta_s2, i_s2, vz0_c_s2, e_s2, omega_s2, P_s2, t_peri_s2])
pos = initial_pos + 1e-6 * np.random.randn(nwalkers, ndim)

sampler = emcee.EnsembleSampler(nwalkers, ndim, log_probability)
sampler.run_mcmc(pos, 1000, progress=True)

flat_samples = sampler.get_chain(discard=300, thin=5, flat=True)
med_beta, med_i, med_vz0, med_e, med_omega, med_P, med_tperi = np.median(flat_samples, axis=0)

# ---------------------------------------------------------
# 5. VALIDATION OUTPUT
# ---------------------------------------------------------
df_truth = pd.read_csv('GR_BLIND_TRUTH.csv').iloc[0]

# Fold the true inclination into the [0, 90] domain for direct mathematical comparison
true_i_rad = np.radians(df_truth['i_true_deg'])
true_i_folded = true_i_rad if true_i_rad <= np.pi/2 else np.pi - true_i_rad

print(f"\n{'='*65}\nFINAL BLIND TEST RESULTS: EXACT R.O.M. GEOMETRY\n{'-'*65}")
print(f"{'Parameter':<25} | {'True (Hidden)':<15} | {'R.O.M. Median':<15}")
print(f"{'Period (yrs)':<25} | {df_truth['P_yrs']:<15.3f} | {med_P/365.25:<15.3f}")
print(f"{'Eccentricity (e)':<25} | {df_truth['e_true']:<15.5f} | {med_e:<15.5f}")
print(f"{'Inclination (folded) [deg]':<25} | {np.degrees(true_i_folded):<15.2f} | {np.degrees(med_i):<15.2f}")
print(f"{'Background Drift (km/s)':<25} | {df_truth['vz0_true']:<15.2f} | {med_vz0*C_KMS:<15.2f}")
print(f"{'-'*65}")
print(f"R.O.M. Kinematic Proj (beta): {med_beta:.6f}")
print(f"{'='*65}")

Loading GR_BLIND_DATASET.csv...
STAGE 1: Keplerian Scout (Locating P, e, T_peri)...
-> Skeleton Found: P = 276.23 yrs, e = 0.9146

STAGE 2: Full R.O.M. Global Sniper (Deterministic Decoupling)...
-> Exact Peak Found: beta = 0.011754, i = 57.10 deg, vz0 = 21.81 km/s

STAGE 3: MCMC Bayesian Mapping (Running 1000 steps)...


100%|██████████| 1000/1000 [00:10<00:00, 95.62it/s]


FINAL BLIND TEST RESULTS: EXACT R.O.M. GEOMETRY
-----------------------------------------------------------------
Parameter                 | True (Hidden)   | R.O.M. Median  
Period (yrs)              | 276.262         | 276.262        
Eccentricity (e)          | 0.91463         | 0.91449        
Inclination (folded) [deg] | 56.99           | 57.09          
Background Drift (km/s)   | 21.14           | 21.82          
-----------------------------------------------------------------
R.O.M. Kinematic Proj (beta): 0.011755


## Phase III: Absolute Metric Extraction
Finally, we translate the dimensionless relational invariants into absolute physical units (meters, seconds, and solar masses) using only the speed of light as a scaling constant.

In [9]:
import numpy as np
import pandas as pd
import os

# ==========================================
# R.O.M. HOLOGRAPHIC EXTRACTOR
# Reconstructing absolute 3D reality from 1D relational data
# ==========================================

C = 299792458.0  # m/s

def reconstruct_system(P_years, e, beta, i_deg, omega_deg, v_z0_kms, data_mode):
    print("="*65)
    print(" 1. INPUT: 1D EXTRACTED RELATIONAL INVARIANTS")
    print("="*65)
    print(f" Period (P):            {P_years:.4f} years")
    print(f" Eccentricity (e):      {e:.5f}")
    print(f" Kinematic Proj (beta): {beta:.6f}")
    print(f" Inclination (i):       {i_deg:.2f} deg")
    print(f" Argument of Peri (w):  {omega_deg:.2f} deg")
    print(f" Background Drift:      {v_z0_kms:.2f} km/s")

    # STAGE 2: METRIC RECOVERY
    P_sec = P_years * 365.25 * 86400.0
    Rs_meters = (P_sec * C * (beta**3)) / np.pi
    a_meters = Rs_meters / (2.0 * beta**2)
    v_p_kms = (beta * np.sqrt((1 + e) / (1 - e)) * C) / 1000.0
    m_rec = (Rs_meters * C**2) / (2 * 6.6743e-11) / 1.98847e30

    print("\n" + "="*65)
    print(" 2. ABSOLUTE METRIC RECOVERY")
    print("="*65)
    print(f" Schwarzschild Radius (Rs): {Rs_meters:,.2f} meters")
    print(f" Semi-major Axis (a):       {a_meters / 1e9:,.4f} million km")
    print(f" Calculated Mass (M_sun):   {m_rec:,.2f}")

    # STAGE 3: HIDDEN TRUTH COMPARISON
    if data_mode == "Synthetic" and os.path.exists('GR_BLIND_TRUTH.csv'):
        truth = pd.read_csv('GR_BLIND_TRUTH.csv').iloc[0]
        true_i_rad = np.radians(truth['i_true_deg'])
        true_i_folded = np.degrees(true_i_rad if true_i_rad <= np.pi/2 else np.pi - true_i_rad)

        print("\n" + "="*65)
        print(" 3. FULL HOLOGRAPHIC VALIDATION (RECOVERED vs. HIDDEN TRUTH)")
        print("="*65)
        print(f" {'Parameter':<25} | {'Recovered':<15} | {'True Value':<15} | {'Error'}")
        print("-"*85)

        params = [
            ("Period (yrs)", P_years, truth['P_yrs'], "%"),
            ("Eccentricity (e)", e, truth['e_true'], "abs"),
            ("Mass (M_sun)", m_rec, truth['M_true_solar'], "%"),
            ("Inclination (deg)", i_deg, true_i_folded, "abs"),
            ("Omega (deg)", omega_deg, truth['w_true_deg'], "abs"),
            ("Bkg Drift (km/s)", v_z0_kms, truth['vz0_true'], "abs")
        ]

        for name, rec, tru, unit in params:
            err = abs(rec - tru) / tru * 100 if unit == "%" else abs(rec - tru)
            err_str = f"{err:.4f}%" if unit == "%" else f"{err:.4f}"
            print(f" {name:<25} | {rec:<15.4f} | {tru:<15.4f} | {err_str}")
        print("="*85)
    else:
        print("\nSTATUS: Custom Data Mode. Hidden truth comparison skipped.")

if __name__ == "__main__":
    try:
        reconstruct_system(
            P_years = med_P / 365.25,
            e = med_e,
            beta = med_beta,
            i_deg = np.degrees(med_i),
            omega_deg = np.degrees(med_omega) % 360,
            v_z0_kms = med_vz0 * 299792.458,
            data_mode = DATA_MODE
        )
    except NameError:
        print("CRITICAL ERROR: Run Phase II (Decoder) before Phase III.")

 1. INPUT: 1D EXTRACTED RELATIONAL INVARIANTS
 Period (P):            276.2617 years
 Eccentricity (e):      0.91449
 Kinematic Proj (beta): 0.011755
 Inclination (i):       57.09 deg
 Argument of Peri (w):  75.98 deg
 Background Drift:      21.82 km/s

 2. ABSOLUTE METRIC RECOVERY
 Schwarzschild Radius (Rs): 1,351,263,718,458.03 meters
 Semi-major Axis (a):       4,889,679.0769 million km
 Calculated Mass (M_sun):   457,537,568.04

 3. FULL HOLOGRAPHIC VALIDATION (RECOVERED vs. HIDDEN TRUTH)
 Parameter                 | Recovered       | True Value      | Error
-------------------------------------------------------------------------------------
 Period (yrs)              | 276.2617        | 276.2615        | 0.0001%
 Eccentricity (e)          | 0.9145          | 0.9146          | 0.0001
 Mass (M_sun)              | 457537568.0367  | 460212415.1173  | 0.5812%
 Inclination (deg)         | 57.0946         | 56.9931         | 0.1015
 Omega (deg)               | 75.9811         | 75.0463 

###NOTICE: G and kg were strictly absent from the generative algebraic chain. They are epistemologically and operationally redundant.


## Conclusion:

###The Universe is invariant where SPACETIME ≡ ENERGY and we call it WILL

[Complete derivation](https://willrg.com/documents/WILL_RG_I.pdf#sec:M_sin(i)) · [Website](https://willrg.com/) · [WILL-AI](https://willrg.com/will-ai/)

